# 🚀 Qwen 3 with Ollama + Ngrok (Fixed Timeout Issues)

This notebook runs **Qwen 3** with **proper warmup** to avoid timeouts.

## Improvements
✅ **Longer timeouts** - 120s for first request
✅ **Model warmup** - Loads model into GPU before testing
✅ **Better error handling**
✅ **Uses qwen3:latest**

---

## Step 1: Check GPU

In [1]:
!nvidia-smi
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB" if torch.cuda.is_available() else "N/A")

Sun Dec  7 15:24:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Install

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pyngrok
print("✓ Installed")

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✓ Installed


## Step 3: Configure Ngrok

In [3]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "2zx1rwuzaVjFIfTotSfdqXHOPSR_ofAcTHk35E86USHnRxmo"
STATIC_DOMAIN = "allegedly-hopeful-stallion.ngrok-free.app"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print(f"✓ Ngrok: {STATIC_DOMAIN}")

✓ Ngrok: allegedly-hopeful-stallion.ngrok-free.app


## Step 4: Start Ollama

In [4]:
import time

print("🚀 Starting Ollama...")
get_ipython().system_raw('OLLAMA_HOST=0.0.0.0:11434 ollama serve > /tmp/ollama.log 2>&1 &')
time.sleep(10)

!ps aux | grep ollama | grep -v grep
print("\n✓ Ollama running")

🚀 Starting Ollama...
root        1237  0.2  0.2 1929932 31464 ?       Sl   15:25   0:00 ollama serve

✓ Ollama running


## Step 5: Pull Qwen 3

In [5]:
print("📥 Pulling qwen3:latest...\n")
!ollama pull qwen3:latest
print("\n✓ Model downloaded")

📥 Pulling qwen3:latest...



✓ Model downloaded


## Step 6: Warmup Model (IMPORTANT!)

**This loads the model into GPU memory to avoid timeouts later.**

In [6]:
import requests

print("🔥 Warming up model (this takes ~30-60 seconds)...")
print("This is normal - first inference loads model into GPU\n")

warmup_url = "http://localhost:11434/api/generate"
warmup_payload = {
    "model": "qwen3:latest",
    "prompt": "Hi",
    "stream": False
}

try:
    response = requests.post(warmup_url, json=warmup_payload, timeout=120)
    if response.status_code == 200:
        print("✓ Model warmed up and ready!")
    else:
        print(f"⚠ Warmup returned status {response.status_code}")
except Exception as e:
    print(f"⚠ Warmup timeout (this is OK, model might still work): {e}")

print("\nWaiting 5 more seconds...")
time.sleep(5)
print("✓ Ready for testing")

🔥 Warming up model (this takes ~30-60 seconds)...
This is normal - first inference loads model into GPU

✓ Model warmed up and ready!

Waiting 5 more seconds...
✓ Ready for testing


## Step 7: Create Tunnel

In [7]:
from IPython.display import display, HTML

OLLAMA_PORT = 11434
print("🌐 Creating ngrok tunnel...")

public_url = ngrok.connect(OLLAMA_PORT, domain=STATIC_DOMAIN)
tunnel_url = f"https://{STATIC_DOMAIN}"

print("\n" + "="*70)
print("✓ READY!")
print("="*70)
print(f"\nAPI Endpoint: {tunnel_url}/v1")
print(f"Model: qwen3:latest")
print(f"\nAdd to .env:")
print(f"  QWEN_BASE_URL={tunnel_url}/v1")
print(f"  QWEN_API_KEY=ollama")
print("="*70)

display(HTML(f'''
<div style="background:#e7f3ff;padding:20px;border-radius:10px;border:2px solid #0066cc">
    <h3 style="color:#0066cc;margin-top:0">🎉 Qwen 3 Ready!</h3>
    <p><strong>Endpoint:</strong> <code>{tunnel_url}/v1</code></p>
    <p><strong>Model:</strong> <code>qwen3:latest</code></p>
</div>
'''))

🌐 Creating ngrok tunnel...

✓ READY!

API Endpoint: https://allegedly-hopeful-stallion.ngrok-free.app/v1
Model: qwen3:latest

Add to .env:
  QWEN_BASE_URL=https://allegedly-hopeful-stallion.ngrok-free.app/v1
  QWEN_API_KEY=ollama


## Step 8: Test Chat (With Longer Timeout)

In [17]:
import json

def test_chat(prompt, timeout=120):
    url = f"http://localhost:{OLLAMA_PORT}/v1/chat/completions"
    payload = {
        "model": "qwen3:latest",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 200
    }

    try:
        response = requests.post(url, json=payload, timeout=timeout)
        response.raise_for_status()
        result = response.json()
        return result['choices'][0]['message']['content']
    except requests.exceptions.Timeout:
        return "ERROR: Timeout - model might be too large for T4 GPU"
    except Exception as e:
        return f"ERROR: {str(e)}"

print("Testing chat (120s timeout)...\n")
response = test_chat("What is 2+2? Answer briefly.")
print(f"Response: {response}\n")

if "ERROR" not in response:
    print("✅ Chat works!")
else:
    print("❌ Chat failed - see troubleshooting below")

Testing chat (120s timeout)...

Response: 

✅ Chat works!


## Step 9: Test Tool Calling

In [19]:
def test_tools():
    url = f"http://localhost:{OLLAMA_PORT}/v1/chat/completions"
    payload = {
        "model": "qwen3:latest",
        "messages": [{"role": "user", "content": "What's the weather in Tokyo?"}],
        "tools": [{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get weather",
                "parameters": {
                    "type": "object",
                    "properties": {"location": {"type": "string"}},
                    "required": ["location"]
                }
            }
        }],
        "tool_choice": "auto"
    }

    try:
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        result = response.json()

        print("Response:")
        print(json.dumps(result, indent=2))

        if 'tool_calls' in result.get('choices', [{}])[0].get('message', {}):
            print("\n" + "="*70)
            print("✅ TOOL CALLING WORKS!")
            print("="*70)
            print("🎉 Can be used for browser automation!")
            return True
        else:
            print("\n" + "="*70)
            print("❌ NO TOOL CALLING")
            print("="*70)
            print("⚠️ Cannot be used for browser automation")
            return False
    except Exception as e:
        print(f"\n❌ Tool test failed: {e}")
        return False

print("🧪 Testing tool calling...\n")
test_tools()

🧪 Testing tool calling...

Response:
{
  "id": "chatcmpl-385",
  "object": "chat.completion",
  "created": 1765125353,
  "model": "qwen3:latest",
  "system_fingerprint": "fp_ollama",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "",
        "reasoning": "Okay, the user is asking for the weather in Tokyo. Let me check the tools available. There's a function called get_weather that requires a location parameter. Since the user mentioned Tokyo, I need to call get_weather with location set to Tokyo. I'll make sure the arguments are correctly formatted as JSON within the tool_call tags.\n",
        "tool_calls": [
          {
            "id": "call_51y97g3d",
            "index": 0,
            "type": "function",
            "function": {
              "name": "get_weather",
              "arguments": "{\"location\":\"Tokyo\"}"
            }
          }
        ]
      },
      "finish_reason": "tool_calls"
    }
  ],
  "usage": 

True

## Troubleshooting

### Still Getting Timeouts?
1. **T4 GPU might be too small** - Qwen 3 might need more VRAM
2. **Try smaller model**: `!ollama pull qwen3:8b` or `qwen3:7b`
3. **Use paid models** - OpenAI/Gemini guaranteed to work
4. **Wait for vLLM** - HuggingFace rate limit clears in 1-2 hours

## Monitor

In [21]:
!ollama list
print(f"\nTunnel: https://{STATIC_DOMAIN}/v1")

NAME            ID              SIZE      MODIFIED    
qwen3:latest    500a1f067a9f    5.2 GB    2 hours ago    

Tunnel: https://allegedly-hopeful-stallion.ngrok-free.app/v1
